In [1]:
import pandas as pd
import numpy as np
import os

In [11]:
name_result = "results_gemini-3.1-pro-preview_python_zero.csv"
file_path = f"../output/results/{name_result}"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print("Arquivo carregado com sucesso!")
else:
    print(f"Erro: O arquivo '{file_path}' não foi encontrado. Verifique se o judge já terminou de rodar.")

Arquivo carregado com sucesso!


In [7]:
df['is_ac'] = df['judge_predict'] == 'AC'
df['test_case_accuracy'] = np.where(df['total_test_cases'] > 0, 
                                    df['AC'] / df['total_test_cases'], 0)

benchmark = df.groupby('level_to_llm').agg(
    qtd_problemas=('question_name', 'count'),
    taxa_sucesso_ac=('is_ac', 'mean'),
    acuracia_testes=('test_case_accuracy', 'mean'),
    tempo_geracao=('llm_code_creation_time', 'mean'),
    tempo_exec_max=('execution_time', 'max'),
    erros_logica=('WA', 'sum'),
    erros_runtime=('RE', 'sum'),
    no_code=('judge_predict', lambda x: (x == 'NO CODE').sum())
).reset_index()

ordem_niveis = {'FACIL': 0, 'MEDIO': 1, 'DIFICIL': 2}
benchmark['ordem'] = benchmark['level_to_llm'].map(ordem_niveis)
benchmark = benchmark.sort_values('ordem').drop('ordem', axis=1)

benchmark['taxa_sucesso_ac'] = (benchmark['taxa_sucesso_ac'] * 100).map('{:.1f}%'.format)
benchmark['acuracia_testes'] = (benchmark['acuracia_testes'] * 100).map('{:.1f}%'.format)
benchmark['tempo_geracao'] = benchmark['tempo_geracao'].map('{:.2f}s'.format)
benchmark['tempo_exec_max'] = benchmark['tempo_exec_max'].map('{:.3f}s'.format)

benchmark.columns = [
    'Nível', 'Qtd', 'Sucesso (AC)', 'Acurácia Testes', 
    'Tempo Geração', 'Gargalo Exec', 'Soma WA', 'Soma RE', 'No Code'
]

print(f"### BENCHMARKING POR NÍVEL DE DIFICULDADE ###")
print(benchmark.to_string(index=False))

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Preparação dos dados
plot_data = benchmark.copy()

# Garante que as colunas sejam numéricas (remove o % se existir)
for col in ['Sucesso (AC)', 'Acurácia Testes']:
    if plot_data[col].dtype == 'object':
        plot_data[col] = plot_data[col].str.replace('%', '').replace('nan', '0').astype(float)

# FORÇAR OS NÍVEIS: Isso garante que 'MEDIO' apareça mesmo que os dados sejam zero
niveis_fixos = ['FACIL', 'MEDIO', 'DIFICIL']
plot_data['Nível'] = pd.Categorical(plot_data['Nível'], categories=niveis_fixos, ordered=True)

# --- Configuração do Gráfico ---
plt.style.use('seaborn-v0_8-muted')
fig, ax1 = plt.subplots(figsize=(10, 6))

# Cores: Verde, Amarelo, Vermelho
colors = ["#2ecc71", "#f1c40f", "#e74c3c"]

# 1. Gráfico de Barras apenas (Linha removida)
# Adicionamos 'hue' para evitar avisos de versões futuras do Seaborn
sns.barplot(
    x='Nível', 
    y='Acurácia Testes', 
    data=plot_data, 
    palette=colors, 
    ax=ax1, 
    edgecolor='black',
    alpha=0.8
)

# Customização
ax1.set_ylim(0, 110)
ax1.set_ylabel('Acurácia nos Testes (%)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Nível de Dificuldade (OBI)', fontsize=12, fontweight='bold')
plt.title('Performance do Agente por Nível', fontsize=15, pad=20, fontweight='bold')

# Adicionar rótulos de dados (Data Labels)
for p in ax1.patches:
    height = p.get_height()
    # Se o valor for NaN ou 0, garantimos que escreva 0.0%
    label_text = f'{height:.1f}%' if not np.isnan(height) else "0.0%"
    ax1.annotate(label_text, 
                 (p.get_x() + p.get_width() / 2., height), 
                 ha='center', va='center', 
                 xytext=(0, 10), 
                 textcoords='offset points',
                 fontweight='bold',
                 fontsize=11)

sns.despine()
plt.tight_layout()

# Salvar opcionalmente
# plt.savefig('benchmark_agente.png', dpi=300)

plt.show()

NameError: name 'benchmark' is not defined

In [9]:
!pip install seaborn

In [10]:
import pandas as pd
import numpy as np

# 1. GARANTIA DE TIPOS: Força as colunas a serem numéricas antes de qualquer matemática
df['AC'] = pd.to_numeric(df['AC'], errors='coerce').fillna(0)
df['total_test_cases'] = pd.to_numeric(df['total_test_cases'], errors='coerce').fillna(0)
df['llm_code_creation_time'] = pd.to_numeric(df['llm_code_creation_time'], errors='coerce').fillna(0)

# 2. CÁLCULOS
df['Taxa de Acerto'] = np.where(
    df['total_test_cases'] > 0, 
    (df['AC'] / df['total_test_cases']) * 100, 
    0.0
)

# 3. FORMATAÇÕES VISUAIS
# Formata a Taxa de Acerto (verifica antes se já é string para evitar erro ao rodar 2x)
df['Taxa de Acerto'] = df['Taxa de Acerto'].apply(
    lambda x: f"{float(x):.1f}%".replace('.', ',') if pd.notna(x) and not isinstance(x, str) else x
)

# Cria a coluna "Total" no formato "Aprovados/Total" (Ex: 52/52)
df['Total'] = df.apply(
    lambda x: f"{int(x['AC'])}/{int(x['total_test_cases'])}" if x['total_test_cases'] > 0 else "—", 
    axis=1
)

# Formata o Tempo de Criação da LLM com 1 casa decimal e o "s" no final
df['Tempo LLM'] = df['llm_code_creation_time'].apply(
    lambda x: f"{float(x):.1f}s" if pd.notna(x) and not isinstance(x, str) else x
)

# Substitui os 0 por "—" (traço) nas métricas de erro para limpar a tabela
# Também garante que os outros erros sejam numéricos antes da checagem
for col in ['AC', 'WA', 'RE', 'TLE', 'CE']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df[col] = df[col].apply(lambda x: "—" if x == 0 else str(int(x)))

# 4. ORGANIZANDO A TABELA FINAL
colunas_finais = [
    'question_name', 'level_to_llm', 'judge_predict', 
    'Taxa de Acerto', 'Total', 'Tempo LLM', 
    'AC', 'TLE', 'WA', 'RE', 'CE'
]
df_final = df[colunas_finais].copy()

# Renomeando as colunas para português (ficando igual ao seu benchmark)
df_final.columns = [
    'Problema', 'Nível', 'Veredito', 
    'Taxa de Acerto', 'Testes (AC/Total)', 'Tempo LLM', 
    'AC', 'TLE', 'WA', 'RE', 'CE'
]

# Ajustando o índice para começar em 1
df_final.index = np.arange(1, len(df_final) + 1)

print(df_final.to_markdown())

ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.